# SROIE 2019 — Data Preprocessing & Exploration
**TrOCR Fine-tuning Project**  
Group: Aziba Mohamed Ayoub · Ballou Moussa · Laidani Med Aymene · Oualid Benchiekh · Amjed Benayad

This notebook covers:
1. Dataset structure inspection
2. Annotation format analysis (bounding boxes + entities)
3. Image-level statistics (resolution, aspect ratio)
4. Text-line statistics (label length, character distribution)
5. Crop generation pipeline
6. Train / Val / Test split and preprocessing for TrOCR

In [ ]:
# ── Cell 0 — Install dependencies ──────────────────────────────────────────
!pip install matplotlib seaborn pandas Pillow -q

In [ ]:
# ── Cell 1 — Imports & paths ────────────────────────────────────────────────
import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from collections import Counter

sns.set_theme(style="whitegrid", palette="muted")
random.seed(42)

BASE      = "/kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019"
TRAIN_IMG = os.path.join(BASE, "train", "img")
TRAIN_BOX = os.path.join(BASE, "train", "box")
TRAIN_ENT = os.path.join(BASE, "train", "entities")
TEST_IMG  = os.path.join(BASE, "test",  "img")
TEST_BOX  = os.path.join(BASE, "test",  "box")

print("Paths configured.")

## 1. Dataset Structure Inspection

In [ ]:
# ── Cell 2 — Count files per split ─────────────────────────────────────────
train_ids = sorted([f[:-4] for f in os.listdir(TRAIN_IMG) if f.endswith(".jpg")])
test_ids  = sorted([f[:-4] for f in os.listdir(TEST_IMG)  if f.endswith(".jpg")])

print(f"Train receipts : {len(train_ids)}")
print(f"Test  receipts : {len(test_ids)}")
print(f"Total receipts : {len(train_ids) + len(test_ids)}")
print("\nSample train IDs:", train_ids[:3])
print("Sample test  IDs:", test_ids[:3])

In [ ]:
# ── Cell 3 — Preview directory tree ────────────────────────────────────────
for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        sub = '  ' * (level + 1)
        for f in files[:2]:
            print(f"{sub}{f}")

## 2. Annotation Format Analysis

In [ ]:
# ── Cell 4 — Box file format (8-point polygon + label) ─────────────────────
sample_id = train_ids[0]
box_path  = os.path.join(TRAIN_BOX, sample_id + ".txt")

print(f"Box file sample: {sample_id}.txt")
print("Format: x1,y1,x2,y2,x3,y3,x4,y4,TEXT\n")
with open(box_path, encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()
for line in lines[:5]:
    print(line.strip())
print(f"\nTotal lines in this receipt: {len(lines)}")

In [ ]:
# ── Cell 5 — Entities file format ──────────────────────────────────────────
ent_path = os.path.join(TRAIN_ENT, sample_id + ".txt")
print(f"Entity file: {sample_id}.txt")
with open(ent_path, encoding="utf-8", errors="ignore") as f:
    print(f.read())

In [ ]:
# ── Cell 6 — Visualise bounding boxes on one receipt ───────────────────────
def parse_box_file(path):
    samples = []
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(",", 8)
            if len(parts) < 9: continue
            try:
                coords = list(map(int, parts[:8]))
            except ValueError:
                continue
            text = parts[8].strip()
            if not text: continue
            xs = coords[0::2]; ys = coords[1::2]
            bbox = (min(xs), min(ys), max(xs), max(ys))
            samples.append((bbox, text))
    return samples

sample_id = train_ids[5]   # pick a different receipt for variety
img = Image.open(os.path.join(TRAIN_IMG, sample_id + ".jpg")).convert("RGB")
boxes = parse_box_file(os.path.join(TRAIN_BOX, sample_id + ".txt"))

fig, ax = plt.subplots(1, 1, figsize=(6, 10))
ax.imshow(img)
for (x1, y1, x2, y2), txt in boxes:
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                              linewidth=1, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
ax.set_title(f"Receipt: {sample_id}\n{len(boxes)} annotated lines", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig("/kaggle/working/sample_receipt_boxes.png", dpi=120, bbox_inches='tight')
plt.show()
print("Saved: sample_receipt_boxes.png")

## 3. Image-Level Statistics

In [ ]:
# ── Cell 7 — Collect image dimensions (train set) ──────────────────────────
widths, heights, aspect_ratios = [], [], []

for rid in train_ids:
    img_path = os.path.join(TRAIN_IMG, rid + ".jpg")
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            aspect_ratios.append(w / h)
    except Exception:
        pass

print(f"Images analysed : {len(widths)}")
print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"Aspect — min: {min(aspect_ratios):.2f}, max: {max(aspect_ratios):.2f}, mean: {np.mean(aspect_ratios):.2f}")

In [ ]:
# ── Cell 8 — Distribution plots for image dimensions ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title("Image Width Distribution")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")

axes[1].hist(heights, bins=30, color='coral', edgecolor='white')
axes[1].set_title("Image Height Distribution")
axes[1].set_xlabel("Height (px)")

axes[2].hist(aspect_ratios, bins=30, color='mediumseagreen', edgecolor='white')
axes[2].set_title("Aspect Ratio Distribution (W/H)")
axes[2].set_xlabel("W / H")

plt.suptitle("SROIE 2019 — Train Image Statistics", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/image_stats.png", dpi=120, bbox_inches='tight')
plt.show()

## 4. Text-Line Statistics

In [ ]:
# ── Cell 9 — Parse all train box files ─────────────────────────────────────
all_train_samples = []
lines_per_receipt = []

for rid in train_ids:
    box_path = os.path.join(TRAIN_BOX, rid + ".txt")
    if not os.path.exists(box_path): continue
    samples = parse_box_file(box_path)
    all_train_samples.extend(samples)
    lines_per_receipt.append(len(samples))

labels = [s[1] for s in all_train_samples]
label_lengths = [len(l) for l in labels]

print(f"Total annotated lines : {len(labels):,}")
print(f"Unique texts          : {len(set(labels)):,}")
print(f"Label length — min: {min(label_lengths)}, max: {max(label_lengths)}, mean: {np.mean(label_lengths):.1f}")
print(f"Lines per receipt — min: {min(lines_per_receipt)}, max: {max(lines_per_receipt)}, mean: {np.mean(lines_per_receipt):.1f}")

In [ ]:
# ── Cell 10 — Label length & lines-per-receipt distributions ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(label_lengths, bins=50, color='slateblue', edgecolor='white')
axes[0].axvline(64, color='red', linestyle='--', label='TrOCR max_length=64')
axes[0].set_title("Text Line Length Distribution")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(lines_per_receipt, bins=30, color='darkorange', edgecolor='white')
axes[1].set_title("Lines per Receipt")
axes[1].set_xlabel("Number of annotated lines")
axes[1].set_ylabel("Count")

plt.suptitle("SROIE 2019 — Text-Line Statistics", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/label_stats.png", dpi=120, bbox_inches='tight')
plt.show()

over_64 = sum(1 for l in label_lengths if l > 64)
print(f"Lines exceeding TrOCR max_length (64 chars): {over_64} ({over_64/len(labels)*100:.2f}%)")

In [ ]:
# ── Cell 11 — Character frequency analysis ─────────────────────────────────
all_chars = Counter(''.join(labels))
top_chars = all_chars.most_common(40)
chars, freqs = zip(*top_chars)

fig, ax = plt.subplots(figsize=(16, 4))
bars = ax.bar(range(len(chars)), freqs, color='teal', edgecolor='white')
ax.set_xticks(range(len(chars)))
ax.set_xticklabels([repr(c)[1:-1] for c in chars], fontsize=9)
ax.set_title("Top 40 Most Frequent Characters in Training Labels", fontsize=13, fontweight='bold')
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig("/kaggle/working/char_freq.png", dpi=120, bbox_inches='tight')
plt.show()

print(f"Total unique characters : {len(all_chars)}")
print(f"Vocabulary size (sorted): {''.join(sorted(all_chars.keys()))}")

In [ ]:
# ── Cell 12 — Crop bounding-box dimensions ─────────────────────────────────
crop_widths, crop_heights, crop_aspects = [], [], []

for rid in random.sample(train_ids, min(100, len(train_ids))):
    img = Image.open(os.path.join(TRAIN_IMG, rid + ".jpg"))
    boxes_info = parse_box_file(os.path.join(TRAIN_BOX, rid + ".txt"))
    for (x1, y1, x2, y2), _ in boxes_info:
        w = x2 - x1; h = y2 - y1
        if w > 0 and h > 0:
            crop_widths.append(w)
            crop_heights.append(h)
            crop_aspects.append(w / h)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(crop_widths,  bins=40, color='steelblue',      edgecolor='white')
axes[0].set_title("Crop Width"); axes[0].set_xlabel("px")

axes[1].hist(crop_heights, bins=40, color='coral',          edgecolor='white')
axes[1].set_title("Crop Height"); axes[1].set_xlabel("px")

axes[2].hist(crop_aspects, bins=40, color='mediumseagreen', edgecolor='white')
axes[2].set_title("Crop Aspect Ratio"); axes[2].set_xlabel("W / H")

plt.suptitle("Text-Line Crop Dimensions (100-receipt sample)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/crop_dims.png", dpi=120, bbox_inches='tight')
plt.show()

print(f"Crop height mean : {np.mean(crop_heights):.1f} px  (receipts vary; most lines ~30-80 px tall)")

## 5. Crop Generation Pipeline

In [ ]:
# ── Cell 13 — Build full crop manifest (train) ─────────────────────────────
OUT_DIR = "/kaggle/working/train_crops"
os.makedirs(OUT_DIR, exist_ok=True)

manifest = []   # list of {img_path, text}
skipped  = 0

for i, rid in enumerate(train_ids):
    img_path = os.path.join(TRAIN_IMG, rid + ".jpg")
    box_path = os.path.join(TRAIN_BOX, rid + ".txt")
    if not os.path.exists(img_path) or not os.path.exists(box_path):
        continue
    img   = Image.open(img_path).convert("RGB")
    boxes = parse_box_file(box_path)
    for j, ((x1, y1, x2, y2), text) in enumerate(boxes):
        # skip degenerate crops
        if x2 - x1 < 10 or y2 - y1 < 5:
            skipped += 1; continue
        crop_path = os.path.join(OUT_DIR, f"{i}_{j}.jpg")
        img.crop((x1, y1, x2, y2)).save(crop_path, quality=90)
        manifest.append({"img_path": crop_path, "text": text})

print(f"Total crops generated : {len(manifest):,}")
print(f"Skipped (too small)   : {skipped}")

In [ ]:
# ── Cell 14 — Visualise 12 random crops ────────────────────────────────────
sample = random.sample(manifest, 12)
fig, axes = plt.subplots(3, 4, figsize=(16, 6))

for ax, item in zip(axes.flatten(), sample):
    img = Image.open(item["img_path"])
    ax.imshow(img, aspect='auto')
    ax.set_title(item["text"][:30], fontsize=8)
    ax.axis('off')

plt.suptitle("12 Random Text-Line Crops from Training Set", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/sample_crops.png", dpi=120, bbox_inches='tight')
plt.show()

## 6. Train / Val / Test Split

In [ ]:
# ── Cell 15 — Deterministic 90/10 train-val split ──────────────────────────
import json

random.shuffle(manifest)
val_size   = int(0.10 * len(manifest))
val_manifest   = manifest[:val_size]
train_manifest = manifest[val_size:]

# Build test manifest (same logic, separate folder)
TEST_OUT_DIR = "/kaggle/working/test_crops"
os.makedirs(TEST_OUT_DIR, exist_ok=True)
test_manifest = []

for i, rid in enumerate(test_ids):
    img_path = os.path.join(TEST_IMG, rid + ".jpg")
    box_path = os.path.join(TEST_BOX, rid + ".txt")
    if not os.path.exists(img_path) or not os.path.exists(box_path): continue
    img   = Image.open(img_path).convert("RGB")
    boxes = parse_box_file(box_path)
    for j, ((x1,y1,x2,y2), text) in enumerate(boxes):
        if x2-x1 < 10 or y2-y1 < 5: continue
        crop_path = os.path.join(TEST_OUT_DIR, f"{i}_{j}.jpg")
        img.crop((x1,y1,x2,y2)).save(crop_path, quality=90)
        test_manifest.append({"img_path": crop_path, "text": text})

print(f"Train  : {len(train_manifest):,} crops")
print(f"Val    : {len(val_manifest):,}   crops")
print(f"Test   : {len(test_manifest):,}  crops")

# Save manifests
with open("/kaggle/working/train_manifest.json", "w") as f: json.dump(train_manifest, f)
with open("/kaggle/working/val_manifest.json",   "w") as f: json.dump(val_manifest, f)
with open("/kaggle/working/test_manifest.json",  "w") as f: json.dump(test_manifest, f)
print("Manifests saved.")

In [ ]:
# ── Cell 16 — Split summary pie chart ──────────────────────────────────────
sizes  = [len(train_manifest), len(val_manifest), len(test_manifest)]
labels = [f'Train\n{sizes[0]:,}', f'Val\n{sizes[1]:,}', f'Test\n{sizes[2]:,}']
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title("Dataset Split — Crop Level", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/split_pie.png", dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 17 — TrOCR preprocessing verification ─────────────────────────────
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

# Show what the processor does to one crop
sample_item = train_manifest[0]
raw_img = Image.open(sample_item["img_path"]).convert("RGB")
pixel_values = processor(images=raw_img, return_tensors="pt").pixel_values
token_ids    = processor.tokenizer(sample_item["text"],
                                    max_length=64, truncation=True).input_ids

print(f"Label text      : {sample_item['text']}")
print(f"Raw crop size   : {raw_img.size}")
print(f"Pixel values    : {pixel_values.shape}  (resized to 384×384, normalised)")
print(f"Token ids       : {token_ids}")
print(f"Decoded tokens  : {processor.tokenizer.decode(token_ids, skip_special_tokens=True)}")

In [ ]:
# ── Cell 18 — Visualise preprocessing: raw vs resized ──────────────────────
import torchvision.transforms.functional as TF
import torch

resized = TF.resize(raw_img, [384, 384])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(raw_img)
axes[0].set_title(f"Original Crop\n{raw_img.size[0]}×{raw_img.size[1]} px", fontsize=11)
axes[0].axis('off')

axes[1].imshow(resized)
axes[1].set_title("After TrOCR Preprocessing\n384×384 px", fontsize=11)
axes[1].axis('off')

plt.suptitle("Preprocessing Example", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/preprocessing_example.png", dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 19 — Final summary table ──────────────────────────────────────────
summary = pd.DataFrame({
    "Split"        : ["Train", "Validation", "Test"],
    "Receipts"     : [len(train_ids), "—", len(test_ids)],
    "Text-Line Crops": [len(train_manifest), len(val_manifest), len(test_manifest)],
    "Used in Training": ["10,000 (subset)", "✓ full", "2,000 (eval subset)"]
})
print(summary.to_string(index=False))

print("\n=== Preprocessing pipeline ===")
print("1. Parse 8-point bounding box → axis-aligned bounding rect")
print("2. Crop PIL image to bounding rect")
print("3. Skip crops < 10 px wide or < 5 px tall")
print("4. TrOCRProcessor: resize to 384×384, normalise pixel values")
print("5. Tokenizer: BPE, max_length=64, truncation=True, padding=True")
print("6. Labels: pad_token_id → -100 (ignored by cross-entropy)")